# Acne improvement, Phase 2: final training (3 seeds), average, threshold, development test once

**What it does.** Trains the chosen configuration with three seeds (42, 7, 123), averages the three models' boxes, picks the confidence threshold on the 154 **validation** photos, and scores the 167-photo development test **once** (stored in a lock file).
**Before you run it:** open Phase 1's result table and set `CONFIG` below to the winning settings (only experiments that passed the +0.02 rule). The default is the baseline recipe (no change).

**Time:** about 2 to 3.5 hours (3 seeds). 

**Before you start (important):**
- Make sure no other training is running (`ps aux | grep -E "nbconvert|lesion_exp"` in WSL). WSL has only about 6.7 GB of memory; two trainings at once can freeze or crash both.
- Kernel: choose the WSL environment `.venv_det` (Python 3.11).
- Run the cells in order. Every finished epoch prints one line; `*` marks a new best mAP@0.5.
- Close other heavy programs (browsers with many tabs, other notebooks) while training.

In [ ]:
import os, sys, json, time, shutil
os.environ["YOLO_VERBOSE"] = "False"          # no progress bars: one clean line per finished epoch is printed instead
from pathlib import Path
sys.path[:0] = ["/mnt/c/Users/Bautista/source/repos/SkinDevApp/" + _d for _d in ("shared", "acne/scripts", "hyperpigmentation/scripts", "eczema/scripts", "skintone", "gradcam")]
HOME = Path.home() / "PrecisionSkinV1"

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import lesion_exp as LE
QUICK_TEST = False                    # True = 1 epoch of seed 42 only, throw-away folder: checks that everything works
TRAIN = True                          # False = only show the results of runs that already finished
ALLOW_RESCORE = False                 # the development test is scored once, then the stored result is shown
# ---- set this after Phase 1 (only changes that passed the rule) ----
CONFIG = dict(name="960_lowlr", recipe="acne_base", over=dict(imgsz=960, batch=6, lr0=0.0005, optimizer="AdamW"))        # = Phase 1 E1: best validation score (+0.027) but recall 0.003 short of the rule; run as a candidate, compared with the installed baseline average in the summary
SEEDS = [42, 7, 123]
IMGSZ = CONFIG["over"].get("imgsz", 800)
TAG = "acne_final_" + CONFIG["name"]
if QUICK_TEST: SEEDS = [42]; TAG += "_QUICKTEST"
VAL_DIR, TEST_DIR = LE.ACNE / "val", LE.ACNE / "test"
LOCK = HOME / "runs" / (TAG + "_devtest_scored.json")
print("configuration:", CONFIG, "| image size", IMGSZ, "| seeds", SEEDS, "| run names start with", TAG)

## 1. Toolbox
Helper functions for boxes, matching, precision / recall / F1, model averaging and plots (nothing here trains).

In [ ]:
import json, csv, os, sys, time, shutil, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
warnings.filterwarnings("ignore")
EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def rd(p):
    p = Path(p); return [tuple(map(float, l.split()[1:5])) for l in p.read_text().splitlines() if len(l.split()) >= 5] if p.exists() else []
def xyxy(b): return b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
def iou(a, b):
    a, b = xyxy(a), xyxy(b); iw, ih = max(0, min(a[2], b[2]) - max(a[0], b[0])), max(0, min(a[3], b[3]) - max(a[1], b[1])); i = iw * ih
    return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-12)
def side(b): return (b[2] * b[3]) ** 0.5
def images_of(d): return sorted(p for p in (Path(d) / "images").glob("*") if p.suffix.lower() in EXT)

def raw_preds(weights, d, imgsz=800):
    """all boxes down to confidence 0.001 for every photo of folder d (normalised x-centre, y-centre, w, h)"""
    from ultralytics import YOLO
    import torch
    m = YOLO(str(weights)); dev = 0 if torch.cuda.is_available() else "cpu"; out = {}; ims = images_of(d)
    for s in range(0, len(ims), 8):
        ch = ims[s:s + 8]
        for p, r in zip(ch, m.predict([str(x) for x in ch], imgsz=imgsz, conf=0.001, iou=0.6, max_det=300, verbose=False, device=dev)):
            out[p.name] = [(tuple(map(float, b)), float(c)) for b, c in zip(r.boxes.xywhn.cpu().numpy(), r.boxes.conf.cpu().numpy())] if len(r.boxes) else []
    return out

def fuse(lists, thr=0.5):
    """weighted box fusion: boxes of the models that overlap (IoU >= thr) become one box; confidence = sum of confidences / number of models"""
    n = len(lists); allb = sorted([b for l in lists for b in l], key=lambda x: -x[1]); clusters = []
    for b, c in allb:
        for cl in clusters:
            if iou(cl["box"], b) >= thr:
                cl["m"].append((b, c)); w = np.array([x[1] for x in cl["m"]]); cl["box"] = tuple(np.average(np.array([x[0] for x in cl["m"]]), axis=0, weights=w)); break
        else: clusters.append({"box": b, "m": [(b, c)]})
    return [(cl["box"], min(1.0, sum(x[1] for x in cl["m"]) / n)) for cl in clusters]

def _ap(conf, tp, n_gt):
    if n_gt == 0 or len(conf) == 0: return 0.0
    o = np.argsort(-conf); tp = tp[o]; tpc = np.cumsum(tp); fpc = np.cumsum(1 - tp); rec = tpc / n_gt; prec = tpc / (tpc + fpc)
    mrec = np.concatenate([[0], rec, [1]]); mpre = np.concatenate([[1], prec, [0]]); mpre = np.flip(np.maximum.accumulate(np.flip(mpre))); x = np.linspace(0, 1, 101)
    return float(np.trapezoid(np.interp(x, mrec, mpre), x)) if hasattr(np, "trapezoid") else float(np.trapz(np.interp(x, mrec, mpre), x))

def match_flags(gt, pr, thr):
    """greedy by confidence: returns (confidences, 1/0 true-positive flags) of every predicted box of one photo"""
    used = set(); cs, fl = [], []
    for b, c in sorted(pr, key=lambda x: -x[1]):
        best, bj = 0, -1
        for j, g in enumerate(gt):
            if j in used: continue
            v = iou(g, b)
            if v > best: best, bj = v, j
        ok = bj >= 0 and best >= thr
        if ok: used.add(bj)
        cs.append(c); fl.append(1.0 if ok else 0.0)
    return cs, fl

def score(preds, d):
    """mAP@0.5, mAP@0.5:0.95 and the raw arrays for precision / recall / F1 at any confidence (ground truth = labels of folder d)"""
    d = Path(d); gts = {k: rd(d / "labels" / (Path(k).stem + ".txt")) for k in preds}; n_gt = sum(len(g) for g in gts.values()); aps = []; conf50 = tp50 = None
    for t in np.arange(0.5, 0.96, 0.05):
        cs, fl = [], []
        for k, pr in preds.items():
            c, f = match_flags(gts[k], pr, t); cs += c; fl += f
        cs, fl = np.array(cs), np.array(fl); aps.append(_ap(cs, fl, n_gt))
        if abs(t - 0.5) < 1e-9: conf50, tp50 = cs, fl
    return dict(AP50=aps[0], AP50_95=float(np.mean(aps)), conf=conf50, tp=tp50, n_gt=n_gt)

def at_conf(s, c):
    m = s["conf"] >= c; tp = s["tp"][m].sum(); p = tp / max(1, m.sum()); r = tp / max(1, s["n_gt"]); return p, r, 2 * p * r / max(1e-9, p + r)

def best_conf(s):
    cs = np.arange(0.05, 0.9, 0.025); return float(max(cs, key=lambda c: at_conf(s, c)[2]))

def rows_at(preds, d, thr):
    """per photo: true positives, false positives, false negatives, zones of your boxes, zones of the model boxes, n your boxes, n model boxes"""
    rows = []
    for k, pr in preds.items():
        gt = rd(Path(d) / "labels" / (Path(k).stem + ".txt")); pc = [b for b, c in pr if c >= thr]; _, fl = match_flags(gt, [(b, 1.0) for b in pc], 0.5); tp = int(sum(fl))
        cell = lambda b: min(2, int(b[1] * 3)) * 3 + min(2, int(b[0] * 3))
        rows.append((tp, len(pc) - tp, len(gt) - tp, {cell(g) for g in gt}, {cell(b) for b in pc}, len(gt), len(pc)))
    return rows

def prf(rows):
    tp, fp, fn = sum(r[0] for r in rows), sum(r[1] for r in rows), sum(r[2] for r in rows); p, r = tp / max(1, tp + fp), tp / max(1, tp + fn); return p, r, 2 * p * r / max(1e-9, p + r)

def zone_f1(rows):
    tp = sum(len(r[3] & r[4]) for r in rows); fp = sum(len(r[4] - r[3]) for r in rows); fn = sum(len(r[3] - r[4]) for r in rows); return 2 * tp / max(1, 2 * tp + fp + fn)

def f1_by_iou(preds, d, thr, ious=(0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8)):
    out = {}
    for t in ious:
        tp = fp = fn = 0
        for k, pr in preds.items():
            gt = rd(Path(d) / "labels" / (Path(k).stem + ".txt")); pc = [(b, 1.0) for b, c in pr if c >= thr]; _, fl = match_flags(gt, pc, t); x = int(sum(fl)); tp += x; fp += len(pc) - x; fn += len(gt) - x
        p, r = tp / max(1, tp + fp), tp / max(1, tp + fn); out[t] = 2 * p * r / max(1e-9, p + r)
    return out

def bootstrap_f1(rows, n=1000, seed=0):
    rng = np.random.default_rng(seed); idx = np.arange(len(rows)); b = [prf([rows[i] for i in rng.choice(idx, len(idx))])[2] for _ in range(n)]; return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

def show_examples(preds, d, thr, names, cols=4, title=""):
    d = Path(d); fig, ax = plt.subplots(-(-len(names) // cols), cols, figsize=(4 * cols, 4 * -(-len(names) // cols))); ax = np.array(ax).reshape(-1)
    for a in ax: a.axis("off")
    for a, n in zip(ax, names):
        p = next(iter((d / "images").glob(Path(n).stem + ".*"))); im = Image.open(p).convert("RGB"); W, H = im.size; dr = ImageDraw.Draw(im); lw = max(2, W // 250)
        for b in rd(d / "labels" / (Path(n).stem + ".txt")): x0, y0, x1, y1 = xyxy(b); dr.rectangle([x0 * W, y0 * H, x1 * W, y1 * H], outline=(40, 220, 60), width=lw)
        for b, c in preds[n]:
            if c >= thr: x0, y0, x1, y1 = xyxy(b); dr.rectangle([x0 * W, y0 * H, x1 * W, y1 * H], outline=(255, 40, 40), width=lw)
        a.imshow(im); a.set_title(n[:28], fontsize=8)
    fig.suptitle(title + "  (green = your boxes, red = model boxes above the threshold)"); plt.tight_layout(); plt.show()

def curves(run_dirs, labels):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for rd_, lb in zip(run_dirs, labels):
        f = Path(rd_) / "results.csv"
        if not f.exists(): continue
        r = pd.read_csv(f); r.columns = [c.strip() for c in r.columns]
        ax[0].plot(r["epoch"], r["train/box_loss"], label=lb + " train"); ax[0].plot(r["epoch"], r["val/box_loss"], "--", label=lb + " val")
        ax[1].plot(r["epoch"], r["metrics/mAP50(B)"], label=lb); ax[2].plot(r["epoch"], r["metrics/mAP50-95(B)"], label=lb)
    for a, t in zip(ax, ("box loss (training = solid, validation = dashed)", "validation mAP@0.5", "validation mAP@0.5:0.95")): a.set_title(t, fontsize=10); a.set_xlabel("epoch"); a.legend(fontsize=7); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 2. Training function

In [ ]:
def train_run(run_id, recipe, over, note, seed, args_base=None, quick=False):
    """one training run with a clean per-epoch line; resumable; returns the run folder"""
    import yaml, torch
    from ultralytics import YOLO
    sp = LE.spec(run_id, "acne", recipe, note, seed=seed, **over); train, vals, _ = LE.recipe(recipe); args = dict(sp["args"]); model = args.pop("model")
    if quick: args["epochs"] = 1
    run = LE.RUNS / run_id; run.mkdir(parents=True, exist_ok=True); lst = run / "train_list.txt"; lst.write_text("\n".join(str(p) for p in train) + "\n")
    first_val = next(iter(vals.values())); yml = run / "data.yaml"
    yaml.safe_dump({"path": str(run), "train": str(lst), "val": str(first_val / "images"), "names": {0: "acne"}}, open(yml, "w")); (run / "spec.json").write_text(json.dumps(sp, indent=1))
    best = {"v": 0.0}
    def on_epoch(trainer):
        m = trainer.metrics; ls = trainer.label_loss_items(trainer.tloss, prefix="train"); mp = m.get("metrics/mAP50(B)", 0.0); star = "*" if mp > best["v"] else " "; best["v"] = max(best["v"], mp)
        print("%s epoch %3d/%d | box %.3f cls %.3f dfl %.3f | P %.3f R %.3f mAP50 %.3f mAP50-95 %.3f" % (star, trainer.epoch + 1, trainer.epochs, ls["train/box_loss"], ls["train/cls_loss"], ls["train/dfl_loss"],
              m.get("metrics/precision(B)", 0), m.get("metrics/recall(B)", 0), mp, m.get("metrics/mAP50-95(B)", 0)), flush=True)
    t0 = time.time(); last = run / "weights" / "last.pt"; print("=" * 110); print(run_id, "|", note, "| %d training images | seed %d" % (len(train), seed), flush=True)
    if (run / "DONE").exists(): print("already trained: using the saved run"); return run
    mdl = YOLO(str(last)) if last.exists() else YOLO(model); mdl.add_callback("on_fit_epoch_end", on_epoch)
    if last.exists(): mdl.train(resume=True)
    else: mdl.train(data=str(yml), seed=seed, device=0 if torch.cuda.is_available() else "cpu", project=str(LE.RUNS), name=run_id, exist_ok=True, plots=False, **args)
    (run / "DONE").write_text("ok"); print("trained in %.1f minutes" % ((time.time() - t0) / 60), flush=True); return run

## 3. Train the three seeds

In [ ]:
RUNS = {}
for s in SEEDS:
    rid = "%s_seed%d" % (TAG, s)
    if TRAIN: train_run(rid, CONFIG["recipe"], CONFIG["over"], "final acne model, configuration %s, seed %d" % (CONFIG["name"], s), seed=s, quick=QUICK_TEST)
    RUNS[s] = LE.RUNS / rid
WEIGHTS = {s: RUNS[s] / "weights" / "best.pt" for s in SEEDS}; print({s: p.exists() for s, p in WEIGHTS.items()})

## 4. Training curves and every finished epoch

In [ ]:
curves([RUNS[s] for s in SEEDS], ["seed %d" % s for s in SEEDS])

In [ ]:
pd.set_option("display.max_rows", None); pd.set_option("display.width", 220)
cols = {"epoch": "epoch", "train/box_loss": "box_loss", "train/cls_loss": "cls_loss", "train/dfl_loss": "dfl_loss", "metrics/precision(B)": "P", "metrics/recall(B)": "R", "metrics/mAP50(B)": "mAP50", "metrics/mAP50-95(B)": "mAP50-95"}
for s in SEEDS:
    f = RUNS[s] / "results.csv"
    if not f.exists(): print("seed", s, ": no results yet"); continue
    r = pd.read_csv(f); r.columns = [c.strip() for c in r.columns]; t = r[list(cols)].rename(columns=cols); t["epoch"] = t["epoch"].astype(int); t["best so far"] = np.where(t["mAP50"] == t["mAP50"].cummax(), "*", "")
    print("=" * 100); print("seed %d | %d finished epochs | best mAP@0.5 %.3f at epoch %d" % (s, len(t), t["mAP50"].max(), int(t.loc[t["mAP50"].idxmax(), "epoch"]))); display(t.round(3).set_index("epoch"))

## 5. Validation results per seed, the 3-model average, and the threshold (chosen on validation only)

In [ ]:
VAL_PRED = {s: raw_preds(WEIGHTS[s], VAL_DIR, IMGSZ) for s in SEEDS}; rows = []
for s in SEEDS:
    sc = score(VAL_PRED[s], VAL_DIR); c = best_conf(sc); p, r, f = at_conf(sc, c); rows.append(dict(seed=s, mAP50=sc["AP50"], mAP50_95=sc["AP50_95"], threshold=c, precision=p, recall=r, F1=f))
vt = pd.DataFrame(rows); display(vt.round(3)); print("mean mAP@0.5 %.3f (std %.3f)" % (vt.mAP50.mean(), vt.mAP50.std() if len(vt) > 1 else 0))
FUSED_VAL = {k: fuse([VAL_PRED[s][k] for s in SEEDS]) for k in VAL_PRED[SEEDS[0]]}; sv = score(FUSED_VAL, VAL_DIR); THR = best_conf(sv); p, r, f = at_conf(sv, THR)
print("3-model average on validation: mAP@0.5 %.3f | mAP@0.5:0.95 %.3f | best F1 %.3f at threshold %.3f (precision %.3f, recall %.3f)" % (sv["AP50"], sv["AP50_95"], f, THR, p, r))
cs = np.arange(0.05, 0.6, 0.025); pr_ = [at_conf(sv, c) for c in cs]
plt.figure(figsize=(6, 3.5)); plt.plot(cs, [x[0] for x in pr_], label="precision"); plt.plot(cs, [x[1] for x in pr_], label="recall"); plt.plot(cs, [x[2] for x in pr_], label="F1", lw=2)
plt.axvline(THR, color="k", ls=":"); plt.xlabel("confidence threshold"); plt.legend(); plt.grid(alpha=.3); plt.title("validation: threshold choice"); plt.show()

## 6. Development test (167 photos), scored once
This set influenced earlier decisions, so treat it as a development number. The independent claim is made in Phase 3 on the fresh test set.

In [ ]:
if LOCK.exists() and not ALLOW_RESCORE:
    R = json.loads(LOCK.read_text()); print("stored result from", R["scored_at"])
else:
    TEST_PRED = {s: raw_preds(WEIGHTS[s], TEST_DIR, IMGSZ) for s in SEEDS}; FUSED_TEST = {k: fuse([TEST_PRED[s][k] for s in SEEDS]) for k in TEST_PRED[SEEDS[0]]}
    st = score(FUSED_TEST, TEST_DIR); rows_t = rows_at(FUSED_TEST, TEST_DIR, THR); p, r, f = prf(rows_t); lo, hi = bootstrap_f1(rows_t); singles = [score(TEST_PRED[s], TEST_DIR)["AP50"] for s in SEEDS]
    R = dict(scored_at=time.strftime("%Y-%m-%d %H:%M"), config=CONFIG["name"], threshold=THR, mAP50=st["AP50"], mAP50_95=st["AP50_95"], precision=p, recall=r, F1=f, F1_ci=[lo, hi], zone_F1=zone_f1(rows_t), single_seed_mAP50=singles,
             n_photos=len(FUSED_TEST), val_mAP50_avg=sv["AP50"])
    LOCK.parent.mkdir(parents=True, exist_ok=True); LOCK.write_text(json.dumps(R, indent=1)); print("scored and stored in", LOCK)
print("DEVELOPMENT TEST (%d photos), 3-model average at threshold %.3f" % (R["n_photos"], R["threshold"]))
print("  mAP@0.5 %.3f | mAP@0.5:0.95 %.3f | precision %.3f | recall %.3f | F1 %.3f (95%% range %.2f-%.2f) | zone F1 %.3f" % (R["mAP50"], R["mAP50_95"], R["precision"], R["recall"], R["F1"], *R["F1_ci"], R["zone_F1"]))
print("  single seeds, mAP@0.5:", [round(x, 3) for x in R["single_seed_mAP50"]], "| installed model before: mAP@0.5 0.439, F1 0.507 (3-seed average, baseline recipe)")

## 7. Export for the app (optional, only after you decide to install)
Off by default. If you turn it on, the files go to `SkinDevApp/Model` and **replace the installed acne models**. The image size must match (960 px models need `imgsz` 960 in the json and the same size in the export). Tell Claude before installing so the app's threshold and the check through the C# code are done too.

In [ ]:
EXPORT = False
if EXPORT:
    from ultralytics import YOLO
    MODEL_DIR = Path("/mnt/c/Users/Bautista/source/repos/SkinDevApp/SkinDevApp/Model")
    for suffix, s in (("", SEEDS[0]), ("_m2", SEEDS[1]), ("_m3", SEEDS[2])):
        o = Path(YOLO(str(WEIGHTS[s])).export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False)); shutil.copy2(o, MODEL_DIR / ("lesion_detector_acne%s.onnx" % suffix)); print("exported seed", s)
    print("now set default_conf = %.3f, imgsz = %d and nms_iou = 0.6 in lesion_detector_acne.json" % (THR, IMGSZ))
else: print("EXPORT = False: nothing written")